In [ ]:
# Automatically fetches required data from sources without manual downloading
# Downloading process
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.signal import savgol_filter
from astropy.io import fits
import requests
import os

# File path configuration
file_path = r'C:\Data\Project\outputs\guncel.txt'

# Directory to save downloaded files
download_dir = r'C:\Data\Project\outputs\indirilen_fits2'
os.makedirs(download_dir, exist_ok=True)

# Directory to save generated plots
plot_dir = r'C:\Data\Project\outputs\grafikler'
os.makedirs(plot_dir, exist_ok=True)

# Read input data
df = pd.read_csv(file_path, sep='\t')

# Function to convert date formats
def convert_date(date_str):
    try:
        # Define expected date formats
        formats = ["%d-%b-%Y", "%d.%m.%Y"]
        for fmt in formats:
            try:
                return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
            except ValueError:
                continue
        raise ValueError(f"Date format not recognized: {date_str}")
    except Exception as e:
        print(f"Error converting date: {e}")
        return None

# File download function
def download_file(url):
    response = requests.head(url)
    if response.status_code == 200:
        local_path = os.path.join(download_dir, url.split('/')[-1])
        if not os.path.exists(local_path):
            response = requests.get(url, stream=True)
            with open(local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024):
                    if chunk:
                        f.write(chunk)
            print(f"Downloaded file: {local_path}")
        else:
            print(f"File already exists: {local_path}")
        return local_path
    else:
        return None
for index, row in df.iterrows():
    try:
        # Convert the date format
        date_str = convert_date(row['Original date'])
        if date_str is None:
            continue  # Skip this row if date conversion fails
        
        sol_format = row['SOL FORMAT']
        first_det = row['first det']
        start_time_str = row['Start (goes)']
        end_time_str = row['End']
        peak_time_str = row['Peak,']
        
        start_time = int(start_time_str.split(':')[0]) * 3600 + int(start_time_str.split(':')[1]) * 60
        end_time = int(end_time_str.split(':')[0]) * 3600 + int(end_time_str.split(':')[1]) * 60
        peak_time = int(peak_time_str.split(':')[0]) * 3600 + int(peak_time_str.split(':')[1]) * 60
        
        year = date_str[:4]
        month = date_str[5:7]
        day = date_str[8:10]
        
        file_versions = ["v03", "v02", "v01", "v00"]  # Versions to try
        file_found = False
        
        for version in file_versions:
            file_url = f"https://heasarc.gsfc.nasa.gov/FTP/fermi/data/gbm/daily/{year}/{month}/{day}/current/glg_ctime_{first_det}_{year[2:]}{month}{day}_{version}.pha"
            local_file_path = download_file(file_url)
            if local_file_path:
                file_found = True
                break

        if not file_found:
            print(f"File not found for row {index}")
            continue
        
        # Check the file size
        if os.path.getsize(local_file_path) < 1024:  # Skip files smaller than 1 KB
            print(f"Downloaded file {local_file_path} is too small, possibly corrupted.")
            continue
        
        title = "{}\nStart: {} End: {}".format(sol_format, start_time_str, end_time_str)
        
        # Light curve plotting
        #plot_light_curve(local_file_path, title, start_time, end_time, peak_time, first_det)
    except Exception as e:
        print(f"Error processing row {index}: {e}")
